# 02 — Feature Engineering & Individual Signal IC

**Goals:**
1. Compute all features from raw price + fundamental + macro data
2. Test each feature's individual IC to understand standalone signal quality
3. Inspect feature correlations — highly correlated features add no marginal information
4. Save the feature matrix and target panel to `data/processed/`

**Key concept:** A feature with IC = 0.0 alone can still add value in a model if it is uncorrelated with other features and captures a distinct return driver. Conversely, adding 5 highly correlated momentum features is not 5x the signal — it may actually hurt by concentrating model weight on one economic factor.

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src.features import build_feature_matrix
from src.targets import build_target_panel, align_features_targets
from src.metrics import information_coefficient, ic_summary

RAW       = cfg.PROJECT_ROOT / cfg.get('data.raw_dir')
PROCESSED = cfg.processed_dir()
PROCESSED.mkdir(parents=True, exist_ok=True)
print('OK')

## 1. Load Raw Data

In [ ]:
prices = pd.read_parquet(RAW / 'prices.parquet')
print(f'Prices: {prices.shape}')

try:
    fundam = pd.read_parquet(RAW / 'fundamentals.parquet')
    print(f'Fundamentals: {fundam.shape} — {fundam.notna().mean().mean():.1%} non-null')
except FileNotFoundError:
    fundam = None
    print('Fundamentals not found — continuing without')

try:
    macro = pd.read_parquet(RAW / 'macro.parquet')
    print(f'Macro: {macro.shape}')
except FileNotFoundError:
    macro = None
    print('Macro not found — continuing without')

## 2. Build Feature Matrix

In [ ]:
features = build_feature_matrix(
    prices=prices,
    fundam=fundam,
    macro=macro,
    cfg_features=cfg.load_config().get('features', {}),
    cross_section_rank=True,  # replace values with pct ranks
)

print(f'Feature matrix: {features.shape}')
print(f'Null rate per feature:')
print(features.isna().mean().sort_values(ascending=False).head(10).to_string())
features.head()

## 3. Build Target Panel

In [ ]:
targets = build_target_panel(
    prices,
    forward_days=cfg.get('target.forward_days', 21),
    n_bins=cfg.get('target.quintile_bins', 5),
)
print(f'Targets: {targets.shape}')
print(targets.describe().T.to_string())

In [ ]:
# Save processed data
features.to_parquet(PROCESSED / 'features.parquet')
targets.to_parquet(PROCESSED / 'targets.parquet')
print(f'Saved to {PROCESSED}')

## 4. Individual Feature IC

In [ ]:
X, y_all = align_features_targets(features, targets)
y_rank = y_all['rank_target']

# Compute IC per feature over the full period
feature_ic = {}
for col in X.columns:
    ic = information_coefficient(X[col].dropna(), y_rank.reindex(X[col].dropna().index))
    feature_ic[col] = ic

ic_series = pd.Series(feature_ic).sort_values(ascending=False)
print('Feature IC (full-period, in-sample — for inspection only):')
print(ic_series.to_string())

In [ ]:
# Visualise
fig, ax = plt.subplots(figsize=(10, max(4, len(ic_series) * 0.3)))
colors = ['#0F6E56' if v > 0 else '#A32D2D' for v in ic_series.values]
ic_series.sort_values().plot(kind='barh', ax=ax, color=colors, alpha=0.8)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('Individual Feature IC — Full Sample (in-sample only, for inspection)')
ax.set_xlabel('Spearman IC')
plt.tight_layout()
plt.show()

print('\n⚠️  These are IN-SAMPLE ICs. Use OOS ICs from notebook 05 for evaluation.')

## 5. Feature Correlation Matrix

In [ ]:
corr = X.corr(method='spearman')

fig, ax = plt.subplots(figsize=(max(8, len(corr) * 0.5), max(8, len(corr) * 0.5)))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.3,
    cbar_kws={'shrink': 0.5},
    annot=len(corr) <= 15,  # only annotate if small enough
    fmt='.2f', annot_kws={'size': 7},
)
ax.set_title('Cross-Sectional Feature Correlation (Spearman)')
plt.tight_layout()
plt.show()

# Flag highly correlated pairs
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        c = corr.iloc[i, j]
        if abs(c) > 0.7:
            high_corr.append({'feat_a': corr.columns[i], 'feat_b': corr.columns[j], 'corr': round(c, 3)})

if high_corr:
    print('Highly correlated pairs (|r| > 0.7) — may be redundant:')
    print(pd.DataFrame(high_corr).to_string(index=False))
else:
    print('No highly correlated pairs found.')

## 6. Rolling IC per Feature (time stability check)

In [ ]:
# Compute rolling per-date IC for key features
key_features = ic_series.head(6).index.tolist()  # top 6 by full-period IC

from src.metrics import rolling_ic

fig, axes = plt.subplots(len(key_features), 1,
                          figsize=(12, 2.5 * len(key_features)), sharex=True)
if len(key_features) == 1:
    axes = [axes]

for ax, feat in zip(axes, key_features):
    ic_ts = rolling_ic(X[feat], y_rank)
    rolling_mean = ic_ts.rolling(6, min_periods=1).mean()
    ax.bar(ic_ts.index, ic_ts.values,
           color=['#0F6E56' if v > 0 else '#A32D2D' for v in ic_ts],
           alpha=0.5, width=15)
    rolling_mean.plot(ax=ax, color='#185FA5', linewidth=1.5)
    ax.axhline(0, color='gray', linewidth=0.6, linestyle='--')
    summary = ic_summary(ic_ts)
    ax.set_title(f'{feat}  |  mean IC={summary["mean_ic"]}  |  ICIR={summary["icir"]}', fontsize=10)

plt.suptitle('Rolling IC per Feature (in-sample — use OOS from nb05 for final eval)',
             fontsize=11, y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary
- Features saved to `data/processed/features.parquet`
- Targets saved to `data/processed/targets.parquet`
- Top features by in-sample IC: (fill in)
- Redundant feature pairs to consider dropping: (fill in)

→ Proceed to `03_baseline_models.ipynb`